# Clase 4: Consultas SQL Avanzadas (JOIN, INSERT, UPDATE, DELETE)

En este laboratorio profundizaremos en el uso de SQL intermedio y avanzado. Usaremos `duckdb` para ejecutar consultas SQL de alto rendimiento directamente sobre archivos CSV y DataFrames.

## 1. Configuración
Instalamos e importamos las librerías necesarias. DuckDB nos permite hacer SQL sobre archivos sin montar una base de datos compleja.

In [ ]:
%pip install duckdb pandas

In [ ]:
import duckdb
import pandas as pd

## 2. Carga de Datos (Baseball)
Descargaremos algunos datos de la base de datos Lahman de béisbol.
- `People.csv`: Información de los jugadores (Nombres, fechas)
- `Batting.csv`: Estadísticas de bateo
- `Teams.csv`: Información de los equipos

In [ ]:
# URLs de los datos
url_people = "https://github.com/chadwickbureau/baseballdatabank/raw/master/core/People.csv"
url_batting = "https://github.com/chadwickbureau/baseballdatabank/raw/master/core/Batting.csv"
url_teams = "https://github.com/chadwickbureau/baseballdatabank/raw/master/core/Teams.csv"

# Creamos la conexión a DuckDB en memoria
con = duckdb.connect(database=':memory:')

# Cargamos los datos directamente desde la URL a tablas de DuckDB
con.execute(f"CREATE TABLE People AS SELECT * FROM read_csv_auto('{url_people}')")
con.execute(f"CREATE TABLE Batting AS SELECT * FROM read_csv_auto('{url_batting}') LIMIT 5000") # Limitamos para rapidez
con.execute(f"CREATE TABLE Teams AS SELECT * FROM read_csv_auto('{url_teams}')")

print("¡Tablas cargadas exitosamente!")

## 3. Repaso de SELECT
Recordemos cómo hacer consultas básicas.

### Ejercicio 3.1: Seleccionar nombres
Selecciona el nombre (`nameFirst`) y apellido (`nameLast`) de todos los jugadores en la tabla `People`.

In [ ]:
# Escribe tu consulta aquí
query = """
SELECT nameFirst, nameLast FROM People LIMIT 10;
"""
con.sql(query).show()

### Ejercicio 3.2: Filtrar por país
Selecciona los jugadores nacidos en 'Venezuela' (`birthCountry`).

In [ ]:
query = """
SELECT nameFirst, nameLast, birthCountry 
FROM People 
WHERE birthCountry = 'Venezuela' 
LIMIT 5;
"""
con.sql(query).show()

## 4. El Poder del JOIN
Las tablas `Batting` y `People` están separadas. Para saber el nombre del jugador junto con sus estadísticas, necesitamos unirlas usando `playerID`.

### Ejemplo 4.1: INNER JOIN
Unir jugadores con sus estadísticas de bateo.

In [ ]:
query = """
SELECT 
    P.nameFirst, 
    P.nameLast, 
    B.yearID, 
    B.HR 
FROM People P
JOIN Batting B ON P.playerID = B.playerID
WHERE B.HR > 30
ORDER BY B.HR DESC
LIMIT 10;
"""
con.sql(query).show()

### Ejemplo 4.2: Uniendo Equipos
Ahora unamos también con la tabla `Teams` para ver el nombre del equipo.

In [ ]:
# Nota: Batting tiene teamID y yearID. Teams tiene teamID y yearID.
# Una unión correcta requiere coincidir ambos.
query = """
SELECT 
    P.nameFirst || ' ' || P.nameLast as Jugador,
    T.name as Equipo,
    B.yearID,
    B.HR
FROM Batting B
JOIN People P ON B.playerID = P.playerID
JOIN Teams T ON B.teamID = T.teamID AND B.yearID = T.yearID
WHERE B.HR > 40
ORDER BY B.HR DESC
LIMIT 10;
"""
con.sql(query).show()

## 5. DML: INSERT, UPDATE, DELETE
Vamos a crear una tabla pequeña de prueba para practicar la modificación de datos sin dañar las tablas principales.

In [ ]:
# Crear tabla vacía basada en People (solo unas columnas)
con.execute("CREATE TABLE MiEquipo (ID VARCHAR, Nombre VARCHAR, Goles INTEGER)")
print("Tabla MiEquipo creada.")

### 5.1 INSERT
Agregamos jugadores nuevos.

In [ ]:
con.execute("INSERT INTO MiEquipo VALUES ('p01', 'Juan Perez', 5)")
con.execute("INSERT INTO MiEquipo VALUES ('p02', 'Maria Gomez', 12)")
con.execute("INSERT INTO MiEquipo VALUES ('p03', 'Carlos Ruiz', 0)")

# Verificamos
con.sql("SELECT * FROM MiEquipo").show()

### 5.2 UPDATE
Carlos metió un gol. Actualicemos su registro.
**IMPORTANTE:** Siempre usa `WHERE`.

In [ ]:
con.execute("UPDATE MiEquipo SET Goles = 1 WHERE ID = 'p03'")

# Verificamos
con.sql("SELECT * FROM MiEquipo").show()

### 5.3 DELETE
Juan se retiró del equipo. Vamos a eliminarlo.

In [ ]:
con.execute("DELETE FROM MiEquipo WHERE ID = 'p01'")

# Verificamos
con.sql("SELECT * FROM MiEquipo").show()

## 6. Tu Turno
Crea una consulta que encuentre a los jugadores que han bateado más de 100 Home Runs (HR) en total en su carrera.
Pista: Necesitarás `GROUP BY playerID` y `SUM(HR)`.

In [ ]:
# Escribe tu respuesta aquí


## 7. Desafíos Progresivos
¡Pongamos a prueba todo lo aprendido! Estos ejercicios van aumentando de dificultad.

### Nivel 1: Calentamiento (SELECT & WHERE)
Encuentra todos los jugadores (`nameFirst`, `nameLast`) que nacieron en 'Mexico' (`birthCountry`) y ordena los resultados por su fecha de nacimiento (`birthYear`, `birthMonth`, `birthDay`).

In [ ]:
# Tu consulta aquí


### Nivel 2: Conexiones (JOIN)
Muestra el nombre del jugador, el nombre de su equipo y el año, para todos los registros donde el jugador haya conectado más de 50 Home Runs (`HR`) en una sola temporada.
- Tablas: `People`, `Batting`, `Teams`

In [ ]:
# Tu consulta aquí


### Nivel 3: Gestión de Equipo (DML)
1. Inserta un nuevo jugador en tu tabla `MiEquipo` con ID 'p99', Nombre 'Tu Nombre' y 10 Goles.
2. Actualiza los goles de 'Tu Nombre' a 20.
3. Muestra la tabla final.

In [ ]:
# Tu código aquí


### Nivel Boss: La Consulta Maestra
Encuentra los **3 equipos** (`name`) que han tenido más jugadores distintos nacidos en 'Venezuela'.
Pista: 
- Unir `People`, `Batting` y `Teams`.
- Filtrar por país.
- Agrupar por nombre del equipo.
- Contar jugadores únicos (`COUNT(DISTINCT playerID)`).
- Ordenar descendente y limitar.

In [ ]:
# ¡Suerte! 
